In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sqlite3

# Task 0
Data extraction: get the data from 3 tables & combine it into single `.csv` file.
After that read this file using pandas to create Dataframe.
So it will be all joined data in 1 dataframe. Quick check - should be 74818 rows in it.

In [ ]:
conn = sqlite3.connect("db.sqlite3")

query = """
SELECT
    o.id AS order_id,
    o.datetime AS order_datetime,
    p.id AS product_id,
    p.name AS product_name,
    p.price AS product_price,
    oi.quantity
FROM order_items oi
JOIN orders o ON oi.order_id = o.id
JOIN products p ON oi.product_id = p.id
"""

df = pd.read_sql_query(query, conn)

conn.close()

print("Rows count:", len(df))

df.to_csv("restaurant_data.csv", index=False)

# Task 1
Get Top 10 most popular products in restaurant sold by Quantity.
Count how many times each product was sold and create a pie chart with percentage of popularity (by quantity) for top 10 of them.

Example:

![pie chart](../demo/pie.png)

In [ ]:
top_products = df.groupby("product_name")["quantity"].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(8, 8))
plt.pie(top_products, labels=top_products.index, autopct='%1.1f%%', startangle=140)
plt.title("Top 10 products by quantity sold")
plt.axis("equal")
plt.show()

# Task 2
Calculate `Item Price` (Product Price * Quantity) for each Order Item in dataframe.
And Make the same Top 10 pie chart, but this time by `Item Price`. So this chart should describe not the most popular products by quantity, but which products (top 10) make the most money for restaurant. It should be also with percentage.

In [ ]:
df["item_price"] = df["product_price"] * df["quantity"]

top_revenue = df.groupby("product_name")["item_price"].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(8, 8))
plt.pie(top_revenue, labels=top_revenue.index, autopct='%1.1f%%', startangle=140)
plt.title("Top 10 products by revenue")
plt.axis("equal")
plt.show()

# Task 3
Calculate `Order Hour` based on `Order Datetime`, which will tell about the specific our the order was created (from 0 to 23). Using `Order Hour` create a bar chart, which will tell the total restaurant income based on the hour order was created. So on x-axis - it will be values from 0 to 23 (hours), on y-axis - it will be the total sum of order prices, which were sold on that hour.

Example:

![bar chart](../demo/bar.png)

In [ ]:
df['order_datetime'] = pd.to_datetime(df['order_datetime'])
df['item_price'] = df['product_price'] * df['quantity']
df['order_hour'] = df['order_datetime'].dt.hour

profit_by_hour = df.groupby('order_hour')['item_price'].sum().reset_index()

plt.figure(figsize=(12, 6))
plt.bar(profit_by_hour['order_hour'], profit_by_hour['item_price'], color='skyblue')
plt.xlabel("Order Hour")
plt.ylabel("Total Profit")
plt.title("Profit by Order Hour")
plt.grid(True, axis='y', linestyle='--', alpha=0.5)
plt.xticks(range(0, 24))
plt.tight_layout()
plt.show()

# Task 4
Make similar bar chart, but right now with `Order Day Of The Week` (from Monday to Sunday), and also analyze total restaurant income by each day of the week.

In [ ]:
df['order_day'] = df['order_datetime'].dt.dayofweek

day_names = {
    0: 'Mon',
    1: 'Tue',
    2: 'Wed',
    3: 'Thu',
    4: 'Fri',
    5: 'Sat',
    6: 'Sun'
}

profit_by_day = df.groupby('order_day')['item_price'].sum().reset_index()
profit_by_day['day_name'] = profit_by_day['order_day'].map(day_names)

profit_by_day = profit_by_day.sort_values(by='order_day')

plt.figure(figsize=(10, 5))
plt.bar(profit_by_day['day_name'], profit_by_day['item_price'], color='orange')
plt.xlabel("Day of the Week")
plt.ylabel("Total Profit")
plt.title("Profit by Day of the Week")
plt.grid(True, axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()